# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import chromadb
from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool


In [3]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
@tool
def retrieve_game(query: str):
    """
    Semantic search: Finds most relevant games in the vector DB
    args:
    - query: a question about game industry.
    
    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    chroma_client = chromadb.PersistentClient(path="chromadb")
    collection = chroma_client.get_collection("udaplay")
    
    results = collection.query(
        query_texts=[query],
        n_results=5
    )
    
    metadatas = results.get('metadatas', [[]])[0] if results else []
    
    return [
        {
            "Platform": m.get("Platform"),
            "Name": m.get("Name"),
            "YearOfRelease": m.get("YearOfRelease"),
            "Description": m.get("Description")
        }
        for m in metadatas
    ]

#### Evaluate Retrieval Tool

In [5]:
import json as _json
from pydantic import BaseModel, Field
from lib.parsers import PydanticOutputParser


class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")


def _to_dict(doc) -> dict:
    """Normalize a doc that may arrive as a dict or a JSON string."""
    if isinstance(doc, dict):
        return doc
    if isinstance(doc, str):
        try:
            parsed = _json.loads(doc)
            return parsed if isinstance(parsed, dict) else {"raw": doc}
        except (_json.JSONDecodeError, ValueError):
            return {"raw": doc}
    return {}


@tool
def evaluate_retrieval(question: str, retrieved_docs: list):
    """
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question.
    args: 
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    docs = [_to_dict(d) for d in retrieved_docs]

    docs_text = "\n".join([
        f"- {d.get('Name', 'Unknown')} ({d.get('Platform', 'Unknown')}, {d.get('YearOfRelease', 'Unknown')}): {d.get('Description', d.get('raw', ''))}"
        for d in docs
    ])

    prompt = f"""Your task is to evaluate if the documents are enough to respond the query.
Give a detailed explanation, so it's possible to take an action to accept it or not.

Query: {question}

Retrieved Documents:
{docs_text}"""

    llm = LLM(api_key=OPENAI_API_KEY)
    response = llm.invoke(prompt, response_format=EvaluationReport)

    parser = PydanticOutputParser(model_class=EvaluationReport)
    report = parser.parse(response)

    return {"useful": report.useful, "description": report.description}

#### Game Web Search Tool

In [6]:
from tavily import TavilyClient


@tool
def game_web_search(question: str):
    """
    Web search: Finds relevant results on the web about the game industry.
    Use this when the vector DB does not have enough information to answer the question.
    args:
    - question: a question about game industry.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(query=question, max_results=5)

    return [
        {
            "title": result.get("title"),
            "url": result.get("url"),
            "content": result.get("content")
        }
        for result in response.get("results", [])
    ]

### Agent

In [7]:
SYSTEM_INSTRUCTIONS = """You are UdaPlay, an expert AI research agent for the video game industry.

Your goal is to answer questions accurately about video games using the following workflow:
1. Use `retrieve_game` to search the internal knowledge base first.
2. Use `evaluate_retrieval` to assess whether the retrieved documents are sufficient to answer the question.
3. If the evaluation returns useful=false, use `game_web_search` to supplement with current web results.
4. Synthesize all available information into a clear, accurate, and concise answer.

Guidelines:
- Always try the internal knowledge base before searching the web.
- Be factual and cite sources when using web results.
- If no information is found anywhere, say so clearly.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=SYSTEM_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.3,
)

In [8]:
questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
]

for question in questions:
    print(f"Q: {question}")
    run = agent.invoke(question)
    final_state = run.get_final_state()
    answer = next(
        (m.content for m in reversed(final_state["messages"]) if isinstance(m, AIMessage) and m.content),
        "No answer found."
    )
    print(f"A: {answer} {'-' * 60}\n")

Q: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
A: Pokémon Gold and Silver were first released in Japan on **November 21, 1999**. They later launched in North America on **October 15, 2000**. These games are notable for introducing the Johto region and new gameplay mechanics, as well as new Pokémon types like Dark and Steel. ------------------------------------------------------------

Q: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processo

### (Optional) Advanced

In [ ]:
from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager
from lib.state_machine import StateMachine, Step, EntryPoint, Termination
from lib.agents import AgentState

import json


# ── Long-term memory setup ──────────────────────────────────────────────────

vector_store_manager = VectorStoreManager(openai_api_key=OPENAI_API_KEY)
long_term_memory = LongTermMemory(db=vector_store_manager)

USER_ID = "udaplay_user"


@tool
def save_to_memory(fact: str):
    """
    Save an important fact or insight to long-term memory for future reference.
    args:
    - fact: a fact or insight worth remembering about the game industry.
    """
    fragment = MemoryFragment(content=fact, owner=USER_ID, namespace="game_facts")
    long_term_memory.register(fragment)
    return {"saved": True, "content": fact}


@tool
def recall_from_memory(query: str):
    """
    Search long-term memory for previously stored facts relevant to the query.
    args:
    - query: a question or topic to search for in long-term memory.
    """
    result = long_term_memory.search(query_text=query, owner=USER_ID, namespace="game_facts", limit=3)
    return [fragment.content for fragment in result.fragments]


def make_tool_node(tool_fn):
    """Wrap a tool into a state machine Step that runs all pending calls for it."""
    def step_logic(state: AgentState) -> AgentState:
        tool_calls = state.get("current_tool_calls") or []
        tool_messages = []

        for call in tool_calls:
            if call.function.name != tool_fn.name:
                continue
            args = json.loads(call.function.arguments)
            result = str(tool_fn(**args))
            tool_messages.append(
                ToolMessage(content=json.dumps(result), tool_call_id=call.id, name=tool_fn.name)
            )

        return {
            "messages": state["messages"] + tool_messages,
            "current_tool_calls": None,
            "session_id": state["session_id"],
        }

    return Step[AgentState](tool_fn.name, step_logic)


extended_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=SYSTEM_INSTRUCTIONS + "\nYou also have access to `save_to_memory` and `recall_from_memory` tools. "
                 "Use `recall_from_memory` at the start of each query, and `save_to_memory` when you learn a new fact.",
    tools=[retrieve_game, evaluate_retrieval, game_web_search, save_to_memory, recall_from_memory],
    temperature=0.3,
)

# Verify the state machine nodes are wired correctly
print("Agent state machine steps:", list(extended_agent.workflow.steps.keys()))
print("Long-term memory ready for user:", USER_ID)

Agent state machine steps: ['__entry__', 'message_prep', 'llm_processor', 'tool_executor', '__termination__']
Long-term memory ready for user: udaplay_user
